In [0]:
from pyspark.sql.functions import *

In [0]:
base_path = "/Volumes/workspace/retail_sales_dw/file_folder"
file_name = "shop_name_20260101.csv"

table_name = "shop_name"

file_path = f"{base_path}/{file_name}"
target_table = f"retail_sales_dw.{table_name}"
target_table_bronze = f"{target_table}_bronze"

header = True
delimiter = ","
format_type = "csv"

In [0]:
#bronze layer
raw_df = (
    spark.read.format(format_type)
    .option("header",header)
    .option("delimiter",delimiter)
    .load(file_path)
    )

raw_df = (
    raw_df
    .withColumn("_load_dt",current_date())
    .withColumn("_load_dttm",current_timestamp())
    .withColumn("_file_name",col("_metadata.file_name"))
    .withColumn("_file_path",col("_metadata.file_path"))
    .withColumn("_file_size",col("_metadata.file_size"))
    .withColumn("_file_mod",col("_metadata.file_modification_time"))
    )
raw_df.display()

In [0]:
# raw_df.write.mode("overwrite").saveAsTable(target_table_bronze)

(
    raw_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table_bronze)
)